# Critical-objective movement model (Google Colab)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joyalzzy/playable-replays/blob/ml/movement-ai.ipynb)

This notebook trains a **separate movement-only LoRA adapter**. Its input contains the controlled player plus all four teammates, including position and alive/dead state, and critical-objective context. Its output contains movement waypoints only.

## Required labels and safety gate

The decoded packet batch alone does not reliably identify team membership, match winner, objective ownership, or whether a player was committed to an objective fight. Supply an enriched, licensed or manually reviewed JSONL export with those labels. Real-data download and all model training are hard-blocked outside Google Colab. Outcome labels select and weight examples but are deliberately omitted from model inputs to prevent label leakage. The bundled schema examples are synthetic and can never enter training.

In [ ]:
from __future__ import annotations

import importlib.util
import hashlib
import json
import platform
import random
import subprocess
import sys
from pathlib import Path
from typing import Any
from urllib.parse import urlparse
from urllib.request import Request, urlopen

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
except ImportError:
    CUDA_AVAILABLE = False

BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  # @param {type:"string"}
DATA_SOURCE = "schema_example"  # @param ["colab_download", "schema_example"]
MOVEMENT_DATA_URL = ""  # @param {type:"string"}
MOVEMENT_DATA_SHA256 = ""  # @param {type:"string"}
MAX_MOVEMENT_DATA_BYTES = 500_000_000
DATA_PATH = Path("/content/playable-replays-output/movement-records.jsonl" if IN_COLAB else "./.local-data/movement-model/schema-only.jsonl")
OUTPUT_DIR = Path("/content/playable-replays-output/movement-model" if IN_COLAB else "./.local-data/movement-model")
OBJECTIVE_WINDOW_SECONDS = 90.0  # @param {type:"number"}
TARGET_POSITIVE_FRACTION = 0.80  # @param {type:"number"}
EVAL_MATCH_FRACTION = 0.20  # @param {type:"number"}
MAX_SEQ_LENGTH = 2048  # @param {type:"integer"}
MAX_STEPS = 60  # @param {type:"integer"}
SEED = 3407  # @param {type:"integer"}
RUN_TRAINING = False  # @param {type:"boolean"}
DOWNLOAD_ARTIFACT = False  # @param {type:"boolean"}

if DATA_SOURCE not in {"colab_download", "schema_example"}:
    raise ValueError("Unsupported DATA_SOURCE")
if DATA_SOURCE == "colab_download" and not IN_COLAB:
    raise RuntimeError("Real movement data may only be downloaded inside Google Colab")
if RUN_TRAINING and not IN_COLAB:
    raise RuntimeError("Movement-model training is restricted to Google Colab")
if OBJECTIVE_WINDOW_SECONDS <= 0:
    raise ValueError("OBJECTIVE_WINDOW_SECONDS must be positive")
if not 0.5 < TARGET_POSITIVE_FRACTION <= 1:
    raise ValueError("TARGET_POSITIVE_FRACTION must be in (0.5, 1]")
if not 0 <= EVAL_MATCH_FRACTION < 1:
    raise ValueError("EVAL_MATCH_FRACTION must be in [0, 1)")
if MAX_SEQ_LENGTH < 256 or MAX_STEPS < 1:
    raise ValueError("Sequence length and training steps are invalid")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps({"python": platform.python_version(), "inColab": IN_COLAB, "cuda": CUDA_AVAILABLE, "outputDir": str(OUTPUT_DIR.resolve())}, indent=2))

## Input contract

Every record needs a stable `matchId`, integer `tick` and `movementIndex`, an alive controlled player, exactly four teammates, and one observed movement target. Alive players require a current `x`/`z` position; a dead teammate may use `null`.

Only allowlisted Baron and dragon-family records labeled `isCritical=true`, `teamCommitted=true`, `fightActive=true`, and within the configured objective window are eligible. Visible enemy state may be supplied for fight context. `matchWonByTeam` and `objectiveSecuredByTeam` must come from reviewed evidence. Split happens by whole match before positive-example weighting, and at least one unused dragon-family match is reserved for evaluation.

In [ ]:
MOVEMENT_PIPELINE_URL = "https://raw.githubusercontent.com/joyalzzy/playable-replays/ml/movement_pipeline.py"
if IN_COLAB and not Path("movement_pipeline.py").is_file():
    request = Request(MOVEMENT_PIPELINE_URL, headers={"User-Agent": "playable-replays-colab-bootstrap/1.0"})
    with urlopen(request, timeout=30) as response:
        module_source = response.read(1_000_001)
    if len(module_source) > 1_000_000:
        raise ValueError("movement_pipeline.py exceeded the bootstrap size cap")
    Path("movement_pipeline.py").write_bytes(module_source)

from movement_pipeline import (
    DRAGON_OBJECTIVE_TYPES,
    TRAINABLE_SOURCE_TYPES,
    balance_training_records,
    build_training_example,
    grouped_split,
    positive_fraction,
    read_records,
    record_order_key,
    select_eligible_records,
    score_movement_prediction,
    summarize_heldout_results,
    validate_record,
    write_jsonl,
)

def schema_record(match_id: str, movement_index: int, *, won: bool, secured: bool, objective_type: str = "baron_nashor", dead_teammate: bool = False) -> dict[str, Any]:
    return {
        "matchId": match_id,
        "tick": 1200,
        "movementIndex": movement_index,
        "state": {
            "controlledPlayer": {"id": "blue-jungle", "alive": True, "position": {"x": 50.0, "z": 50.0}},
            "teammates": [
                {"id": "blue-top", "alive": True, "position": {"x": 46.0, "z": 51.0}},
                {"id": "blue-mid", "alive": not dead_teammate, "position": None if dead_teammate else {"x": 51.0, "z": 48.0}},
                {"id": "blue-marksman", "alive": True, "position": {"x": 48.0, "z": 47.0}},
                {"id": "blue-support", "alive": True, "position": {"x": 49.0, "z": 49.0}},
            ],
            "visibleEnemies": [
                {"id": "red-jungle", "alive": True, "position": {"x": 55.0, "z": 55.0}},
                {"id": "red-support", "alive": True, "position": {"x": 57.0, "z": 54.0}},
            ],
            "objective": {"type": objective_type, "position": {"x": 56.0, "z": 57.0}, "isCritical": True, "teamCommitted": True, "fightActive": True, "secondsToResolution": 25.0},
        },
        "label": {
            "movement": {"waypoints": [{"x": 53.0, "z": 54.0}, {"x": 55.0, "z": 56.0}]},
            "matchWonByTeam": won,
            "objectiveSecuredByTeam": secured,
            "sourceType": "schema_example",
            "evidenceId": f"schema-{match_id}-{movement_index}",
        },
        "metadata": {"coordinateMethod": "synthetic x/z schema coordinates", "labelProvenance": "synthetic schema example; never train"},
    }

SCHEMA_EXAMPLES = [
    schema_record("schema-win", 0, won=True, secured=True),
    schema_record("schema-win", 1, won=True, secured=True, dead_teammate=True),
    schema_record("schema-loss", 0, won=False, secured=False, objective_type="elder_dragon"),
    schema_record("schema-dragon", 0, won=True, secured=True, objective_type="dragon"),
    schema_record("schema-excluded", 0, won=True, secured=True, objective_type="rift_herald"),
]

def download_movement_records() -> Path:
    if not IN_COLAB:
        raise RuntimeError("Movement training data may only be downloaded inside Google Colab")
    parsed_url = urlparse(MOVEMENT_DATA_URL)
    if parsed_url.scheme != "https" or not parsed_url.netloc:
        raise ValueError("MOVEMENT_DATA_URL must be a non-empty HTTPS URL")
    if len(MOVEMENT_DATA_SHA256) != 64 or any(character not in "0123456789abcdefABCDEF" for character in MOVEMENT_DATA_SHA256):
        raise ValueError("MOVEMENT_DATA_SHA256 must be a 64-character SHA-256 digest")
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    partial_path = DATA_PATH.with_name(DATA_PATH.name + ".part")
    digest = hashlib.sha256()
    total_bytes = 0
    request = Request(MOVEMENT_DATA_URL, headers={"User-Agent": "playable-replays-colab-movement/1.0"})
    with urlopen(request, timeout=60) as response, partial_path.open("wb") as output:
        while chunk := response.read(1024 * 1024):
            total_bytes += len(chunk)
            if total_bytes > MAX_MOVEMENT_DATA_BYTES:
                raise ValueError("Movement dataset exceeded the configured download cap")
            digest.update(chunk)
            output.write(chunk)
    actual_sha256 = digest.hexdigest()
    if actual_sha256.lower() != MOVEMENT_DATA_SHA256.lower():
        raise ValueError(f"Movement dataset checksum mismatch: {actual_sha256}")
    partial_path.replace(DATA_PATH)
    print(f"Downloaded and verified {total_bytes:,} bytes in Google Colab")
    return DATA_PATH

raw_records = read_records(download_movement_records()) if DATA_SOURCE == "colab_download" else SCHEMA_EXAMPLES
records = [validate_record(record, index) for index, record in enumerate(raw_records)]
records.sort(key=record_order_key)
positions = [record_order_key(record) for record in records]
if len(positions) != len(set(positions)):
    raise ValueError("Duplicate (matchId, tick, movementIndex) positions are not allowed")
eligible_records, exclusion_counts = select_eligible_records(records, objective_window_seconds=OBJECTIVE_WINDOW_SECONDS)
trainable_records = [record for record in eligible_records if record["label"]["sourceType"] in TRAINABLE_SOURCE_TYPES]
raw_train_records, eval_records = grouped_split(trainable_records, eval_fraction=EVAL_MATCH_FRACTION, seed=SEED, required_eval_objective_types=DRAGON_OBJECTIVE_TYPES)
train_match_ids = {record["matchId"] for record in raw_train_records}
eval_match_ids = {record["matchId"] for record in eval_records}
if train_match_ids & eval_match_ids:
    raise AssertionError("Held-out evaluation matches overlap training matches")
train_records = balance_training_records(raw_train_records, target_positive_fraction=TARGET_POSITIVE_FRACTION, seed=SEED) if raw_train_records else []

if RUN_TRAINING:
    if DATA_SOURCE != "colab_download":
        raise ValueError("Synthetic schema examples are forbidden when RUN_TRAINING=True")
    if len({record['matchId'] for record in trainable_records}) < 2 or not eval_records:
        raise ValueError("Training requires at least two labeled matches and a held-out match group")
    if len(train_records) < 2:
        raise ValueError("Training requires at least two selected movement examples")
    if not any(record["state"]["objective"]["type"] in DRAGON_OBJECTIVE_TYPES and record["state"]["objective"]["fightActive"] for record in eval_records):
        raise ValueError("Held-out evaluation requires an unused dragon-family objective fight")

train_examples = [build_training_example(record) for record in train_records]
eval_examples = [build_training_example(record) for record in eval_records]
preview_examples = train_examples or [build_training_example(record) for record in eligible_records]
write_jsonl(OUTPUT_DIR / "train.jsonl", train_examples)
write_jsonl(OUTPUT_DIR / "eval.jsonl", eval_examples)
write_jsonl(OUTPUT_DIR / "eligible.preview.jsonl", [build_training_example(record) for record in eligible_records[:20]])
selection_report = {
    "inputRecords": len(records),
    "eligibleCriticalObjectiveRecords": len(eligible_records),
    "excluded": exclusion_counts,
    "trainableBeforeSplit": len(trainable_records),
    "rawTrainRecords": len(raw_train_records),
    "weightedTrainRecords": len(train_records),
    "weightedPositiveFraction": positive_fraction(train_records),
    "naturalEvalRecords": len(eval_records),
    "naturalEvalPositiveFraction": positive_fraction(eval_records),
    "heldOutDragonFightRecords": sum(record["state"]["objective"]["type"] in DRAGON_OBJECTIVE_TYPES for record in eval_records),
}
(OUTPUT_DIR / "selection-report.json").write_text(json.dumps(selection_report, indent=2), encoding="utf-8")
print(json.dumps(selection_report, indent=2))
if DATA_SOURCE == "schema_example":
    print("Schema examples validated and were excluded from training. Use colab_download with labeled data to train.")

In [ ]:
if not preview_examples:
    raise ValueError("No eligible critical-objective movement examples were found")
preview = preview_examples[0]
print("MODEL INPUT (outcome labels intentionally absent):")
print(json.dumps(json.loads(preview["messages"][1]["content"]), indent=2))
print("\nMOVEMENT TARGET:")
print(json.dumps(json.loads(preview["messages"][2]["content"]), indent=2))

## Train the separate QLoRA adapter

In Google Colab, set `DATA_SOURCE=colab_download`, provide the HTTPS URL and SHA-256 of the enriched movement JSONL, and inspect `selection-report.json`. Then assign a CUDA runtime and set `RUN_TRAINING=True`. Local data download and local training are blocked.

In [ ]:
if RUN_TRAINING:
    if not IN_COLAB:
        raise RuntimeError("Training dependencies may only be installed for this workflow in Google Colab")
    if not CUDA_AVAILABLE:
        raise RuntimeError("QLoRA training requires an assigned CUDA GPU runtime")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "unsloth", "datasets", "trl"])
else:
    print("Dependency installation skipped (RUN_TRAINING=False).")

In [ ]:
model = tokenizer = trainer = None
training_metrics = None
if RUN_TRAINING:
    from datasets import Dataset
    from transformers import TrainingArguments
    from trl import SFTTrainer
    from unsloth import FastLanguageModel, is_bfloat16_supported

    model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(
        model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth", random_state=SEED, use_rslora=False,
    )

    def add_text(batch: dict[str, list[Any]]) -> dict[str, list[str]]:
        return {"text": [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False) for messages in batch["messages"]]}

    train_dataset = Dataset.from_list(train_examples).map(add_text, batched=True)
    eval_dataset = Dataset.from_list(eval_examples).map(add_text, batched=True)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_dataset, eval_dataset=eval_dataset, dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=1, packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=min(5, max(1, MAX_STEPS // 10)), max_steps=MAX_STEPS,
            learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), logging_steps=1, eval_strategy="steps", eval_steps=max(1, MAX_STEPS // 5),
            optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear", seed=SEED, output_dir=str(OUTPUT_DIR / "checkpoints"), report_to="none",
        ),
    )
    training_result = trainer.train()
    training_metrics = training_result.metrics
    print(json.dumps(training_metrics, indent=2, default=str))
else:
    print("Movement model training skipped.")

## Test on unused major-objective fights

Every test example comes from a match ID absent from training. The report measures JSON validity and final-waypoint error overall, by objective type, and specifically for dragon-family fights. This is an offline held-out imitation test, not live gameplay evaluation.

In [ ]:
heldout_results: list[dict[str, Any]] = []
heldout_report = None
if RUN_TRAINING:
    FastLanguageModel.for_inference(model)
    for example in eval_examples:
        prompt_messages = example["messages"][:2]
        input_ids = tokenizer.apply_chat_template(prompt_messages, add_generation_prompt=True, tokenize=True, return_tensors="pt").to(model.device)
        attention_mask = torch.ones_like(input_ids)
        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=192, do_sample=False, use_cache=True)
        generated_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()
        try:
            prediction = json.loads(generated_text)
        except json.JSONDecodeError:
            prediction = generated_text
        expected = json.loads(example["messages"][2]["content"])
        scored = score_movement_prediction(prediction, expected)
        scored.update({
            "matchId": example["metadata"]["matchId"],
            "tick": example["metadata"]["tick"],
            "objectiveType": example["metadata"]["objectiveType"],
            "fightActive": example["metadata"]["fightActive"],
            "livingTeammates": example["metadata"]["livingTeammates"],
            "positiveSelectionClass": example["metadata"]["positiveSelectionClass"],
            "rawOutput": prediction,
        })
        heldout_results.append(scored)
    write_jsonl(OUTPUT_DIR / "heldout-predictions.jsonl", heldout_results)
    heldout_report = summarize_heldout_results(heldout_results)
    if heldout_report["dragonObjectiveFights"]["examples"] < 1:
        raise AssertionError("Held-out report contains no dragon-family objective fight")
    (OUTPUT_DIR / "heldout-report.json").write_text(json.dumps(heldout_report, indent=2), encoding="utf-8")
    print(json.dumps(heldout_report, indent=2))
else:
    print("Held-out generation test runs after Colab training. Match split and dragon-holdout gates were validated.")

## Export and interpretation boundary

Report endpoint error separately for won-and-secured versus contrast examples, by objective type, and by number of living teammates. Positive training weighting is intentional imitation bias; it does not show that the learned movement caused a win or objective capture.

In [ ]:
import shutil
from datetime import datetime, timezone

if RUN_TRAINING:
    adapter_dir = OUTPUT_DIR / "critical-objective-movement-lora"
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    manifest = {
        "createdAt": datetime.now(timezone.utc).isoformat(),
        "baseModel": BASE_MODEL,
        "method": "QLoRA movement-only supervised fine-tuning",
        "criticalObjectiveTypes": ["baron_nashor", "dragon", "elemental_dragon", "dragon_soul", "elder_dragon"],
        "objectiveWindowSeconds": OBJECTIVE_WINDOW_SECONDS,
        "targetPositiveFraction": TARGET_POSITIVE_FRACTION,
        "actualPositiveFraction": positive_fraction(train_records),
        "trainExamples": len(train_examples),
        "evalExamples": len(eval_examples),
        "seed": SEED,
        "trainingMetrics": training_metrics,
        "heldoutEvaluation": heldout_report,
        "inputContract": "controlled player plus exactly four teammate position/alive states and critical-objective context",
        "labelLeakageControl": "match and objective outcomes select/weight examples but are absent from model prompts",
        "causalityDisclosure": "Selection favors observed wins with secured objectives; it does not establish that movement caused either outcome.",
    }
    (adapter_dir / "training-manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
    archive = shutil.make_archive(str(OUTPUT_DIR / "critical-objective-movement-lora"), "zip", root_dir=adapter_dir)
    print(f"Saved movement adapter: {archive}")
    if DOWNLOAD_ARTIFACT:
        if not IN_COLAB:
            raise RuntimeError("Browser download is available only in Colab")
        from google.colab import files
        files.download(archive)
else:
    print(f"Selection artifacts are available in {OUTPUT_DIR.resolve()}; adapter export skipped.")

## Run with real data

1. Host a reviewed movement JSONL at an HTTPS URL. Team IDs, active-fight context, objective ownership, and match outcome must be explicit labels backed by `evidenceId` and `metadata.labelProvenance`.
2. Open this notebook in Google Colab. Set `DATA_SOURCE=colab_download`, `MOVEMENT_DATA_URL`, and the pinned `MOVEMENT_DATA_SHA256`.
3. Run through the preview and inspect `selection-report.json`. Confirm the match-disjoint test set includes dragon-family fights.
4. Assign a Colab CUDA runtime, set `RUN_TRAINING=True`, and rerun.
5. Inspect `heldout-report.json` and `heldout-predictions.jsonl` before using the adapter. Do not combine its movement output with the packet-prediction adapter.